# Atrasos e avaliações

Como as avaliações variam entre entregas no prazo e atrasadas? Compara notas e faixas de atraso.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)


In [ ]:
from pathlib import Path

# Procura a raiz a partir da pasta atual; funciona também nas cópias em outputs/notebooks.
RAIZ_PROJETO = next(
    (pasta for pasta in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (pasta / 'executar_pipeline.py').is_file() and (pasta / 'src').is_dir()),
    None
)
if RAIZ_PROJETO is None:
    raise FileNotFoundError('Execute o notebook dentro da pasta do projeto ou de uma subpasta.')

CAMINHO_PROCESSADOS = RAIZ_PROJETO / 'data' / 'processed'

df_base = pd.read_parquet(f'{CAMINHO_PROCESSADOS}/base_analitica_pedidos.parquet')


In [ ]:
colunas_analise = [
    'order_id',
    'review_score',
    'review_ruim',
    'review_neutra',
    'review_boa',
    'possui_avaliacao',
    'pedido_entregue',
    'delivery_delay_days',
    'is_late'    
]

df_entregas = (
    df_base[colunas_analise]
    .loc[lambda dados: dados['pedido_entregue'] & dados['possui_avaliacao']]
    .dropna(subset=['delivery_delay_days', 'is_late'])
    .copy()
)

print(df_entregas.isna().sum())


In [ ]:
df_comparacao_atraso = pd.DataFrame({
    'status_entrega': ['atrasado', 'no_prazo'],
    'total_pedidos': [
        df_entregas.loc[lambda dados: dados['is_late']].shape[0],
        df_entregas.loc[lambda dados: ~dados['is_late']].shape[0]
    ],
    'reviews_ruins': [
        df_entregas.loc[lambda dados: dados['is_late'] & dados['review_ruim']].shape[0],
        df_entregas.loc[lambda dados: ~dados['is_late'] & dados['review_ruim']].shape[0]
    ],
    'reviews_neutras': [
        df_entregas.loc[lambda dados: dados['is_late'] & dados['review_neutra']].shape[0],
        df_entregas.loc[lambda dados: ~dados['is_late'] & dados['review_neutra']].shape[0]
    ],
    'reviews_boas': [
        df_entregas.loc[lambda dados: dados['is_late'] & dados['review_boa']].shape[0],
        df_entregas.loc[lambda dados: ~dados['is_late'] & dados['review_boa']].shape[0]
    ],
    'nota_media': [
        df_entregas.loc[lambda dados: dados['is_late'], 'review_score'].mean(),
        df_entregas.loc[lambda dados: ~dados['is_late'], 'review_score'].mean()
    ]
})

for tipo in ['ruins', 'neutras', 'boas']:
    df_comparacao_atraso[f'pct_reviews_{tipo}'] = (
        df_comparacao_atraso[f'reviews_{tipo}'] / df_comparacao_atraso['total_pedidos']
    )
    
display(df_comparacao_atraso.style.format({
    'pct_reviews_ruins': '{:.2%}',
    'pct_reviews_neutras': '{:.2%}',
    'pct_reviews_boas': '{:.2%}',
    'nota_media': '{:.2f}'
}))


In [ ]:
df_pedidos_atrasados = df_entregas.loc[df_entregas['is_late']].copy()


labels = [
    'ate_3_dias',
    'de_4_a_7_dias',
    'de_8_a_14_dias',
    '15_dias_ou_mais'
]

df_pedidos_atrasados['faixa_atraso'] = pd.cut(
    df_pedidos_atrasados['delivery_delay_days'],
    bins=[0, 3, 7, 14, float('inf')],
    labels=labels,
    include_lowest=True    
)

df_atrasos_por_faixa = (
    df_pedidos_atrasados
    .groupby('faixa_atraso', observed=True)
    .agg(
        quantidade=('order_id', 'count'),
        nota_media=('review_score', 'mean'),
        reviews_ruins=('review_ruim', 'sum'),
        reviews_neutras=('review_neutra', 'sum'),
        reviews_boas=('review_boa', 'sum'),
    )
)

for tipo in ('ruins', 'neutras', 'boas'):
    df_atrasos_por_faixa[f'pct_reviews_{tipo}'] = (
        df_atrasos_por_faixa[f'reviews_{tipo}']
        / df_atrasos_por_faixa['quantidade']
    )
    
display(
    df_atrasos_por_faixa.style.format({
        'nota_media': '{:.2f}',
        'pct_reviews_ruins': '{:.2%}',
        'pct_reviews_neutras': '{:.2%}',
        'pct_reviews_boas': '{:.2%}'
    })
)


## Gráficos

Montagem dos gráficos da análise.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from io import BytesIO

plt.rcParams['text.parse_math'] = False
PASTA_GRAFICOS = RAIZ_PROJETO / 'outputs' / 'graficos'
graficos_exportar = {}


def registrar_grafico(figura, caminho, dpi=180, bbox_inches='tight'):
    # Captura antes de fechar a figura; grava os arquivos apenas na seção final.
    buffer = BytesIO()
    figura.savefig(buffer, format='png', dpi=dpi, bbox_inches=bbox_inches)
    nome = Path(caminho).name
    if nome in graficos_exportar:
        raise ValueError(f'Nome de gráfico repetido: {nome}')
    graficos_exportar[nome] = buffer.getvalue()


def concluir_grafico(figura, nome, nota):
    figura.text(0.01, 0.01, nota, fontsize=9, color='#555555')
    figura.tight_layout(rect=[0, 0.09, 1, 0.97])
    registrar_grafico(figura, nome)
    plt.show()
    plt.close(figura)


def grafico_situacoes(tabela, grupo, metrica, denominador, titulo, nome, nota):
    # Ausência de grupo permanece NaN: não representa uma taxa observada de zero.
    grupos = tabela[grupo].drop_duplicates().tolist()
    fig, ax = plt.subplots(figsize=(15, max(5, len(grupos) * 0.62 + 2)))
    cores = {'Atrasado': '#C95050', 'No prazo': '#457B9D', 'Sem informação': '#888888'}
    situacoes = [s for s in cores if s in tabela['situacao_entrega'].unique()]
    largura = 0.75 / max(1, len(situacoes))
    for i, situacao in enumerate(situacoes):
        dados = tabela.loc[tabela['situacao_entrega'].eq(situacao)].set_index(grupo).reindex(grupos)
        taxas = dados[metrica].astype(float)
        barras = ax.barh(
            [j + (i - (len(situacoes) - 1) / 2) * largura for j in range(len(grupos))],
            taxas, height=largura, color=cores[situacao], label=situacao
        )
        rotulos = [
            f'{t:.2%}'.replace('.', ',') + f' (n={int(n):,})'.replace(',', '.')
            if pd.notna(t) and pd.notna(n) else ''
            for t, n in zip(taxas, dados[denominador])
        ]
        ax.bar_label(barras, labels=rotulos, padding=3, fontsize=8)
    ax.set_yticks(range(len(grupos)), [str(g).replace('_', ' ') for g in grupos])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.28)
    ax.set_xticks([0, .2, .4, .6, .8, 1])
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.set_xlabel('Proporção')
    ax.set_title(titulo, loc='left', pad=35)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1), ncol=3, frameon=False)
    ax.grid(axis='x', alpha=.2)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    concluir_grafico(fig, nome, nota)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter


In [ ]:
df_grafico = (
    df_comparacao_atraso
    .set_index('status_entrega')
    .loc[
        ['no_prazo','atrasado'],
        ['pct_reviews_boas', 'pct_reviews_neutras', 'pct_reviews_ruins']
    ]
    .rename(
        index={'no_prazo' : 'No prazo', 'atrasado' : 'Atrasado'},
        columns={'pct_reviews_boas' : 'Boas', 'pct_reviews_neutras' : 'Neutras', 'pct_reviews_ruins' : 'Ruins'}
    )
)


In [ ]:
# Cria a figura e a área onde o gráfico será desenhado.
# fig: a figura completa, incluindo gráfico e textos.
# ax: a área do gráfico, onde ficam barras, eixos e legenda.
# figsize: largura e altura em polegadas.
fig, ax = plt.subplots(figsize=(9, 5))

# Cria barras agrupadas usando as colunas do df_grafico.
df_grafico.plot.bar(
    ax=ax,  # Desenha na área criada acima.
    color=['#2A9D8F', '#E9C46A', '#C95050'],  # Cores das colunas, na ordem.
    width=0.75,  # Largura ocupada pelo grupo de barras.
    rot=0  # Mantém os textos do eixo horizontal sem rotação.
)

# Define o título do gráfico.
ax.set_title(
    'Pedidos atrasados apresentam maior proporção de avaliações ruins',
    loc='left',  # Alinha o título à esquerda.
    pad=16  # Espaço entre o título e a área do gráfico.
)

# Remove o título do eixo horizontal.
# Os nomes "No prazo" e "Atrasado" continuam aparecendo.
ax.set_xlabel('')

# Define o título do eixo vertical.
ax.set_ylabel('Proporção dos pedidos')

# Exibe os valores do eixo vertical como porcentagens.
# xmax=1 significa que 1 representa 100% e 0.5 representa 50%.
# Isso altera a exibição, não os valores dos dados.
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))

# Define os limites do eixo vertical: de 0% a 100%.
ax.set_ylim(0, 1)

# Adiciona uma legenda com título e sem borda.
ax.legend(title='Avaliação', frameon=False)

# Adiciona linhas horizontais suaves para facilitar a leitura.
# alpha controla a transparência: 0 é invisível e 1 é opaco.
ax.grid(axis='y', alpha=0.2)

# Coloca as linhas da grade atrás das barras.
ax.set_axisbelow(True)

# Remove as bordas superior e direita da área do gráfico.
ax.spines[['top', 'right']].set_visible(False)

# Percorre os conjuntos de barras: boas, neutras e ruins.
for barras in ax.containers:

    # Cria um texto percentual para cada barra do conjunto.
    rotulos = [
        # get_height(): obtém o valor representado pela barra.
        # .1%: formata como porcentagem com uma casa decimal.
        # replace(): troca o ponto decimal por vírgula.
        # Exemplo: 0.6285 vira "62,8%".
        f'{barra.get_height():.1%}'.replace('.', ',')
        for barra in barras
    ]

    # Escreve os percentuais acima das barras deste conjunto.
    # Este comando fica dentro do for para rotular todos os conjuntos.
    ax.bar_label(
        barras,
        labels=rotulos,
        padding=3,  # Distância entre o topo da barra e o texto.
        fontsize=9  # Tamanho da fonte.
    )

# Busca a quantidade de pedidos de cada situação.
# O resultado permite acessar, por exemplo, contagens['no_prazo'].
contagens = (
    df_comparacao_atraso
    .set_index('status_entrega')['total_pedidos']
)

# Define os textos abaixo dos dois grupos de barras.
# A ordem precisa ser a mesma usada no df_grafico.
# \n: quebra a linha.
# n: quantidade de pedidos do grupo.
# int(): transforma a contagem em número inteiro.
# :, separa os milhares com vírgula; replace troca por ponto.
ax.set_xticklabels([
    f"No prazo\n(n = {int(contagens['no_prazo']):,})".replace(',', '.'),
    f"Atrasado\n(n = {int(contagens['atrasado']):,})".replace(',', '.')
])

# Adiciona uma nota à figura inteira.
# As coordenadas vão de 0 a 1:
# (0, 0) é o canto inferior esquerdo;
# (1, 1) é o canto superior direito.
fig.text(
    0.01,  # Posição horizontal, próxima da borda esquerda.
    0.01,  # Posição vertical, próxima da borda inferior.
    'Recorte: pedidos entregues, avaliados e com prazo conhecido. '
    'Boas: 4–5; neutras: 3; ruins: 1–2.',
    fontsize=9,
    color='#555555'  # Cor cinza.
)

# Ajusta os espaços para reduzir sobreposições.
# rect = [esquerda, base, direita, topo].
# A base em 0.07 reserva espaço na parte inferior para a nota.
fig.tight_layout(rect=[0, 0.07, 1, 1])

# Exibe a figura no notebook.
plt.show()


In [ ]:
from pathlib import Path

# Usa a mesma pasta de saída da execução pelo pipeline.
# Reutiliza a raiz identificada na célula de leitura dos dados.

PASTA_GRAFICOS = RAIZ_PROJETO / 'outputs' / 'graficos'


registrar_grafico(fig, 
    PASTA_GRAFICOS / '02_avaliacoes_por_situacao_entrega.png',
    dpi=300,
    bbox_inches='tight'
)


## Exportações

Salva as tabelas em Parquet e CSV e os gráficos em PNG.

In [ ]:
PASTA_PARQUET = RAIZ_PROJETO / 'outputs' / 'tabelas' / 'parquet'
PASTA_CSV = RAIZ_PROJETO / 'outputs' / 'tabelas' / 'csv'
for pasta in (PASTA_PARQUET, PASTA_CSV, PASTA_GRAFICOS):
    pasta.mkdir(parents=True, exist_ok=True)

tabelas_exportar = {
    '02_comparacao_atraso': df_comparacao_atraso,
    '02_atrasos_por_faixa': df_atrasos_por_faixa.reset_index(),
}
for nome, original in tabelas_exportar.items():
    tabela = original.reset_index(drop=True).convert_dtypes()
    caminho = PASTA_PARQUET / f'{nome}.parquet'
    tabela.to_parquet(caminho, index=False)
    pd.testing.assert_frame_equal(tabela, pd.read_parquet(caminho), check_exact=True)
    tabela.to_csv(
        PASTA_CSV / f'{nome}.csv', index=False,
        sep=';', decimal=',', encoding='utf-8-sig'
    )
    print(f'Tabela exportada em Parquet e CSV: {nome} ({len(tabela)} linhas)')

for nome, conteudo in graficos_exportar.items():
    (PASTA_GRAFICOS / nome).write_bytes(conteudo)
    print(f'Gráfico exportado: {nome}')
